--------------------------------------------------------------------------------
# **CZECH NEURO-SYMBOLIC DATASET TRANSFORMER (STANZA VERSION)**

--------------------------------------------------------------------------------

This tutorial/script demonstrates how to build a high-performance token-tagging
pipeline for the Czech language without using flaky or unstable Generative LLMs
or broken Hugging Face model checkpoints. Instead of forcing an LLM to 
hallucinate, we leverage the official Stanza NLP engine developed by Stanford 
University, pre-trained on the Prague Dependency Treebank (PDT).

How it works:
1. **Data Ingestion:**
   - The script streams 100% of rows from 5 distinct Polars Parquet datasets 
     categorized by punctuation type (comma, exclamation_mark, none, period, question_mark).
2. **Deep Linguistic Token Classification:**
   - Every input segment is analyzed via a state-of-the-art neural pipeline.
     Instead of simple linear tags, it extracts true morphological features and
     syntactic dependency trees.
3. **Symbolic Extraction Layer:**
   - It extracts exact Part-of-Speech tags, numerical Czech grammatical cases 
     (1. až 7. pád) directly from morphological attributes, and precise sentence 
     roles (podmet, prisudek, predmet).
4. **Safe 1:1 Alignment:**
   - Guarantees absolute deterministic word-to-tag alignment and outputs a clean,
     production-ready JSON array for each dataset without mid-sentence cuts.

### ***Prerequisites & Installation:***
------------------------------------------
To run this script locally, execute the following command in your terminal:

```cmd
    pip install stanza polars pyarrow tqdm
```

No Hugging Face token or external authentication is required.

### ***Output:***
---------------------
The script creates a directory named 'gold_datasets_output/' and exports 5 clean, 
fully annotated JSON files ready for your nanoGPT embedding layer.


In [1]:
!pip install stanza polars pyarrow tqdm -qqq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 19.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 19.4 MB/s eta 0:00:00


In [2]:
import os
import json
import polars as pl
from tqdm import tqdm
import stanza

import torch

### ***1. DATASET CONFIGURATION***

In [3]:
BASE_DATASET_DIR = os.path.join("/kaggle","input","notebooks","radimkzl","czech-text-dataset","raw_segments_dataset")  # Root folder, where you have the comma, period, etc. folders.
OUTPUT_GOLD_DIR = os.path.join("/kaggle","working","gold_datasets_output")   # Output folder for finished JSONs

### ***Map of all 5 of your Parquet datasets***

In [4]:
DATASET_MAP = {
    "comma": os.path.join(BASE_DATASET_DIR, "comma", "part-0.parquet"),
    "exclamation_mark": os.path.join(BASE_DATASET_DIR, "exclamation_mark", "part-0.parquet"),
    "none": os.path.join(BASE_DATASET_DIR, "none", "part-0.parquet"),
    "period": os.path.join(BASE_DATASET_DIR, "period", "part-0.parquet"),
    "question_mark": os.path.join(BASE_DATASET_DIR, "question_mark", "part-0.parquet")
}

### ***Mapping UPOS tags to your names***

In [5]:
POS_MAP = {
    "NOUN": "podstatne_jmeno",
    "VERB": "sloveso",
    "AUX": "sloveso",
    "PRON": "zajmeno",
    "DET": "zajmeno",
    "ADV": "prislovce",
    "ADP": "predlozka"
}

### ***Mapping sentence elements (Stanza knows real dependency syntax!)***

In [6]:
DEPREL_MAP = {
    "nsubj": "podmet",
    "root": "prisudek",
    "obj": "predmet",
    "iobj": "predmet"
}

## ***2. CORE PIPELINE***

In [7]:
def process_single_dataset(punc_key, parquet_path, nlp_pipeline):
    if not os.path.exists(parquet_path):
        print(f"⚠️ Warning: File for '{punc_key}' not found. Skipping.")
        return

    print(f"\n Loading COMPLETE dataset [{punc_key}] from: {parquet_path}")
    df = pl.read_parquet(parquet_path)

    raw_data = (
        df.select(["segment", "punctuation_type"])
        .filter(pl.col("segment").is_not_null())
        .to_dicts()
    )

    gold_dataset = []
    
    for row in tqdm(raw_data, desc=f"⏳ Stanza Tagging -> {punc_key}"):
        segment_text = row["segment"]
        punc_type = row["punctuation_type"]
        
        try:
            # Running analysis via Stanza
            doc = nlp_pipeline(segment_text)
            tokens_annotation = []
            
            for sentence in doc.sentences:
                for word in sentence.words:
                    # We omit punctuation
                    if word.upos == "PUNCT":
                        continue
                        
                    slovni_druh = POS_MAP.get(word.upos, "jiny")
                    vetny_clen = DEPREL_MAP.get(word.deprel, "jiny")
                    
                    # Extracting the fall from morphological features (feats)
                    # Stanza returns feats in the format "Case=Nom|Gender=Fem|..."
                    pad = 0
                    if word.feats and "Case=" in word.feats:
                        for feat in word.feats.split("|"):
                            if feat.startswith("Case="):
                                case_val = feat.split("=")[1]
                                # Mapping of falls to numbers (Nom=1, Gen=2, Dat=3, Acc=4, Voc=5, Loc=6, Ins=7)
                                case_map = {"Nom": 1, "Gen": 2, "Dat": 3, "Acc": 4, "Voc": 5, "Loc": 6, "Ins": 7}
                                pad = case_map.get(case_val, 0)

                    tokens_annotation.append({
                        "slovo": word.text,
                        "slovni_druh": slovni_druh,
                        "vetny_clen": vetny_clen,
                        "pad": pad
                    })
                    
            gold_dataset.append({
                "segment": segment_text,
                "punctuation_type": punc_type,
                "tokens_annotation": tokens_annotation
            })

        except Exception:
            continue

    output_path = os.path.join(OUTPUT_GOLD_DIR, f"gold_dataset_{punc_key}.json")
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(gold_dataset, f, ensure_ascii=False, indent=2)
        
    print(f"💾 Dataset [{punc_key}] saved successfully.")

## ***3. MAIN START-UP***

In [8]:
if __name__ == "__main__":
    print("🔬 Initializing Local Research Pipeline (STANZA CZECH MODE)")
    print("---------------------------------------------------------------------------")
    
    os.makedirs(OUTPUT_GOLD_DIR, exist_ok=True)
    
    # Download the official Czech model (it is downloaded from the Stanford repositories, not from HF)
    print("⏳ Downloading/Verifying the official Czech package for Stanza...")
    stanza.download('cs', processors='tokenize,pos,lemma,depparse', verbose=False)
    
    # Pipeline initialization (pos = parts of speech + cases, depparse = clauses)
    nlp_pipeline = stanza.Pipeline('cs', processors='tokenize,pos,lemma,depparse', verbose=False, use_gpu=True)
    print("The Stanza model is ready.")

    for punc_key, parquet_path in DATASET_MAP.items():
        process_single_dataset(punc_key, parquet_path, nlp_pipeline)
        
    print("\n---------------------------------------------------------------------------")
    print(f"🎉 ALL DATASETS HAVE BEEN COMPLETELY PROCESSED!")

🔬 Initializing Local Research Pipeline (STANZA CZECH MODE)
---------------------------------------------------------------------------
⏳ Downloading/Verifying the official Czech package for Stanza...
The Stanza model is ready.

 Loading COMPLETE dataset [comma] from: /kaggle/input/notebooks/radimkzl/czech-text-dataset/raw_segments_dataset/comma/part-0.parquet


⏳ Stanza Tagging -> comma: 100%|██████████| 5036/5036 [05:48<00:00, 14.46it/s]


💾 Dataset [comma] saved successfully.

 Loading COMPLETE dataset [exclamation_mark] from: /kaggle/input/notebooks/radimkzl/czech-text-dataset/raw_segments_dataset/exclamation_mark/part-0.parquet


⏳ Stanza Tagging -> exclamation_mark: 100%|██████████| 22/22 [00:01<00:00, 14.83it/s]


💾 Dataset [exclamation_mark] saved successfully.

 Loading COMPLETE dataset [none] from: /kaggle/input/notebooks/radimkzl/czech-text-dataset/raw_segments_dataset/none/part-0.parquet


...

💾 Dataset [none] saved successfully.

 Loading COMPLETE dataset [period] from: /kaggle/input/notebooks/radimkzl/czech-text-dataset/raw_segments_dataset/period/part-0.parquet


⏳ Stanza Tagging -> period: 100%|██████████| 4752/4752 [05:55<00:00, 13.38it/s]


💾 Dataset [period] saved successfully.

 Loading COMPLETE dataset [question_mark] from: /kaggle/input/notebooks/radimkzl/czech-text-dataset/raw_segments_dataset/question_mark/part-0.parquet


⏳ Stanza Tagging -> question_mark: 100%|██████████| 171/171 [00:11<00:00, 15.02it/s]

💾 Dataset [question_mark] saved successfully.

---------------------------------------------------------------------------
🎉 ALL DATASETS HAVE BEEN COMPLETELY PROCESSED!
